# 16 · 50 行手撸 Naive RAG

> **学习目标**：把 RAG 6 步骨架（load → chunk → embed → store → retrieve → generate）用最少代码跑通。提供 OFFLINE/ONLINE 双模式，**只换一行代码**就能从离线 stub 切到本机 Ollama。
>
> **预备**：Foundations Stage 1 + Stage 2 走完（09 mini_vecdb、08 qwen_sampling 都见过）。
>
> **为什么重要**：RAG 框架（langchain / llama-index / ragflow）说穿了都是这 6 步的包装。先把骨架手写一遍，看库源码就不再迷糊。

In [ ]:
MODE = 'OFFLINE'        # 'OFFLINE' or 'ONLINE'（ONLINE 前请先 `ollama serve`）

import numpy as np, requests, hashlib, time
from typing import Callable
np.set_printoptions(precision=3, suppress=True)
print(f'MODE = {MODE}')

## 1. 双模式抽象 —— embed + chat 两个函数搞定

**OFFLINE**：embed 用 hash 派生的确定性向量，chat 用「拼 context + 简单模板」的伪 LLM。
**ONLINE**：embed 走 Ollama `/api/embeddings`，chat 走 `/api/chat`。

**关键设计**：两种模式共享同一调用签名 → 上层 RAG 代码完全不用改。

In [ ]:
OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text: str, dim: int = 256) -> np.ndarray:
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    return np.random.default_rng(seed).standard_normal(dim).astype(np.float32)

def ollama_embed(text: str, model: str = 'nomic-embed-text') -> np.ndarray:
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model': model, 'prompt': text}, timeout=30)
    r.raise_for_status()
    return np.array(r.json()['embedding'], dtype=np.float32)

def fake_chat(prompt: str) -> str:
    # 提取 prompt 里的「问题」与「相关片段」做最朴素拼装
    return f'[FAKE-LLM] 基于上下文我能告诉你: {prompt[-120:]!s}'

def ollama_chat(prompt: str, model: str = 'qwen1.5_1.8') -> str:
    r = requests.post(f'{OLLAMA}/api/chat', json={
        'model': model, 'stream': False, 'options': {'temperature': 0.2},
        'messages': [{'role': 'user', 'content': prompt}],
    }, timeout=120)
    r.raise_for_status()
    return r.json()['message']['content']

if MODE == 'ONLINE':
    try:
        requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status()
        embed: Callable[[str], np.ndarray] = ollama_embed
        chat:  Callable[[str], str]         = ollama_chat
        print('✅ Ollama 在线，使用真实 embedding / LLM')
    except Exception as e:
        print(f'⚠ Ollama 未启动 ({type(e).__name__})，自动降级到 OFFLINE 模式')
        MODE = 'OFFLINE'
if MODE == 'OFFLINE':
    embed = fake_embed
    chat  = fake_chat
    print('使用 OFFLINE stub（hash-based embedding + 拼接式假 LLM）')

## 2. 6 步流程逐个写

**Step 1: 准备 documents**（这里直接 inline 5 段中文。生产中是从 PDF/DB 来的）

In [ ]:
DOCS = [
    'Transformer 是 2017 年 Google 提出的基于注意力机制的神经网络架构，是现代大模型的基础。',
    'RAG（Retrieval-Augmented Generation）通过把外部知识库的检索结果拼进上下文，缓解大模型幻觉问题。',
    'LoRA 是一种参数高效的微调方法，在原模型权重旁附加低秩矩阵，训练时只更新这些低秩矩阵。',
    '向量数据库（如 Chroma、Qdrant、Milvus）通过 ANN 算法（如 HNSW）加速高维向量的相似度检索。',
    'Chinchilla 论文（DeepMind 2022）给出了「参数量与训练 token 数最优配比」的 scaling law。',
]
for i, d in enumerate(DOCS):
    print(f'[{i}] {d}')

In [ ]:
# Step 2: chunk —— 5 段都很短，直接每段一个 chunk
def chunk(text: str, max_len: int = 200) -> list[str]:
    if len(text) <= max_len:
        return [text]
    # 按句号粗粒度切，每段不超过 max_len
    sentences = text.split('。')
    out, buf = [], ''
    for s in sentences:
        if not s.strip():
            continue
        s = s + '。'
        if len(buf) + len(s) > max_len and buf:
            out.append(buf); buf = s
        else:
            buf += s
    if buf:
        out.append(buf)
    return out

chunks = []
for doc_id, doc in enumerate(DOCS):
    for c in chunk(doc):
        chunks.append({'doc_id': doc_id, 'text': c})
print(f'共 {len(chunks)} 个 chunk')
for c in chunks[:3]:
    print(' ', c)

In [ ]:
# Step 3 + 4: embed + store（合一，存储就是 numpy 数组）
vectors = np.vstack([embed(c['text']) for c in chunks])
# 归一化（详见 notebook 20）—— 让查询时点积 = cosine 相似度
vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12)
print('vectors shape:', vectors.shape, 'dtype:', vectors.dtype)

In [ ]:
# Step 5: retrieve
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    q = embed(query)
    q = q / (np.linalg.norm(q) + 1e-12)
    sims = vectors @ q
    order = np.argsort(-sims)[:top_k]
    return [{**chunks[i], 'score': float(sims[i])} for i in order]

# 演示
for hit in retrieve('什么是 LoRA？', top_k=3):
    print(f"  doc{hit['doc_id']}  sim={hit['score']:+.3f}  {hit['text'][:50]}...")

In [ ]:
# Step 6: generate —— 构造 prompt 把 context 拼进去
PROMPT_TEMPLATE = '''你是一个严谨的助手。仅基于以下「上下文」回答用户问题。
如果上下文不足以回答，请明确说「不知道」。

上下文：
{context}

问题：{question}
回答：'''

def rag(question: str, top_k: int = 3) -> str:
    hits = retrieve(question, top_k=top_k)
    context = '\n'.join(f'[doc{h["doc_id"]}] {h["text"]}' for h in hits)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    return chat(prompt)

for q in ['什么是 RAG？', 'LoRA 跟微调什么关系？', '向量库怎么加速？']:
    print(f'\nQ: {q}')
    print(f'A: {rag(q)[:200]}')

## 3. 全流程一句话总结

```python
# 离线索引
vectors = normalize([embed(c) for c in chunks(docs)])

# 在线查询
def rag(q):
    hits = top_k(vectors, embed(q))
    return chat(prompt_template(context=hits, question=q))
```

**整个 RAG 系统的核心就是这两行**。剩下都是工程优化（chunking 策略、embedding 选型、混合检索、re-rank、评估、监控……），在后续 notebook 一个个展开。

## 4. 失败模式 —— 看 fake_embed 在哪里出洋相

**OFFLINE 模式的 fake_embed 是 hash-based**：完全没有语义，只有「完全相同字符串 → 相同向量」。所以**只有 query 与 chunk 字符串高度重合时才能召回**。这就是为什么真实 RAG 必须用语义 embedding。

In [ ]:
# OFFLINE 模式下，看「同义改写」会不会被召回
test_queries = [
    ('Transformer 是什么？',  '应能命中 doc0（包含 "Transformer"）'),
    ('注意力网络的发明？',     'fake_embed 命中不到（同义改写）'),
    ('LoRA',                  '应能命中 doc2'),
    ('参数高效微调',          'fake_embed 命中不到（同义改写）'),
]
for q, expect in test_queries:
    hit = retrieve(q, top_k=1)[0]
    print(f'  {q!r:25} -> doc{hit["doc_id"]} (sim={hit["score"]:+.3f})  [{expect}]')

if MODE == 'OFFLINE':
    print('\n→ OFFLINE 模式下「同义改写」必然不命中 —— 因为 fake_embed 没有语义。')
    print('→ 切到 ONLINE（ollama serve + nomic-embed-text）后，同义改写会命中。')

## 深入思考

1. **为什么写入时归一化？查询时也归一化？**
   - 归一化后 `cosine(q, d) = q · d`（避免每次查询都除两个范数）。详见 notebook 20。
2. **为什么 `chunk()` 用 `。` 分割？换英文怎么办？**
   - 这是最朴素的分句。真实场景用 langchain 的 `RecursiveCharacterTextSplitter`（详见 notebook 19）会自动按 `\n\n` → `\n` → `。` → `？` 等多级 separator 切。
3. **PROMPT_TEMPLATE 里的「仅基于上下文」「不知道就说不知道」起作用吗？**
   - 部分起作用。**真正可靠的拒答**还需要：(a) 设 `temperature=0`；(b) 在 context 前后加 `<context>` 标签；(c) 训练数据 / 后训练 / 评估闭环都要有 "don't make up" 信号。
4. **`top_k=3` 不够怎么办？**
   - 不一定要变大。先看是「召回不准」还是「召回准但答案差」，前者改检索（hybrid / rerank），后者改 prompt / 改模型。
5. **这个 50 行能扩到 100 万文档吗？**
   - 不能。Brute-force 检索 O(N)，10⁶ 量级 → 慢。要上 HNSW（Chroma / Qdrant 都自带）。**但所有逻辑骨架不变**，只是 `vectors @ q` 换成 `index.query(q)`。

**改一改**：
- 把 `MODE` 改成 `'ONLINE'`（确保 `ollama serve` 在跑），看「同义改写」是不是变得能命中
- 把 `top_k` 改成 1，看答案质量
- 加一段 query → 故意问个 docs 里完全没有的事，看模型会不会胡说

## 自检 ✅

- [ ] 画出 RAG 6 步流程图，每步说出输入/输出 shape
- [ ] 解释「为什么 OFFLINE 的 fake_embed 同义改写召不到」
- [ ] 默写检索的 4 行核心代码（embed q → normalize → 矩阵乘 → argsort）
- [ ] 给一份失败的 RAG，能讲清「下一步该排查 chunking / embedding / retrieval / prompt 哪一段」
- [ ] 不查文档写出 PROMPT_TEMPLATE 的 3 个最关键约束（仅基于上下文 / 拒答 / 输出格式）

## 下一步

→ [`17_rag_project_walkthrough.ipynb`](17_rag_project_walkthrough.ipynb)